<a href="https://colab.research.google.com/github/ounissinour/PolypVision/blob/main/milestone2/M2_Improve.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 2 – Improve: retrain Polyp-PVT with a proper validation split

**PolypVision** · Nour Ounissi (RSPN69), Taha Mersni (K1TO74), Baha Amar (ZXQA2V)

**Why retrain at all?** The official `Train.py` of Polyp-PVT scores the five *test* sets after every
epoch and saves the epoch with the best test Dice as `PolypPVT.pth`. The released model is therefore
chosen on the test data, and its test score is optimistic. Here the model is chosen on a **validation
split** of the training data (made in Milestone 1 by `prepare_data.py`); the test sets are used once, at
the end, for the chosen checkpoint.

| Step | What happens |
|---|---|
| 1 | Settings, packages, official Polyp-PVT code (pinned commit), our `polyp_m2.py` and `prepare_data.py` |
| 2 | Download training + test data and weights (re-uses the Milestone 1 Drive cache if it is there) |
| 3 | Train / validation / test split (identical to Milestone 1) |
| 4 | Checks: our inference + metric give the paper numbers with the authors' model; the loss as the official code really computes it; augmentation preview |
| 5 | Experiments, one factor at a time, 30 epochs each: **loss functions**, **augmentations**, **image sizes** |
| 6 | Results: validation vs test, training curves, effect of choosing the epoch on the test set |
| 7 | Final model: best option per factor (chosen on validation), 100 epochs, compared with the official recipe, the authors' model and PraNet |
| 8 | Optional: MedSAM with bounding-box prompts |

**Time.** Every run saves its state to Drive after each epoch and continues where it stopped after a
disconnect, and finished runs are skipped. So you can simply *Run all* again in a new session. To share
the work, give each team member a few experiments with `RUN_ONLY` and copy the finished run folders
(small: `done.json`, `history.csv`, `test_*.csv`, `best.pth`) into one `runs/` folder before section 6.

## 1. Settings

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

`EPOCHS` is the length of every ablation run, `FINAL_EPOCHS` of the two final runs (100 is the
official schedule). `RUN_ONLY = ["baseline", "loss_dice"]` trains only those experiments.

In [ ]:
import os

PVT_COMMIT = "cd71fbf9fcd8c32cc0f33699f4e0186073c64dc4"      # official Polyp-PVT repo, as in Milestone 1
MEDSAM_COMMIT = "d71e8a1a99ad751840a22a7fa3ecfb4166fb1488"   # official MedSAM repo
REPO_RAW = "https://raw.githubusercontent.com/ounissinour/PolypVision/main"

USE_DRIVE = True
EPOCHS = 30            # each ablation run
FINAL_EPOCHS = 100     # the final runs
RUN_ONLY = None        # e.g. ["baseline", "aug_flip"]; None = all experiments
RUN_FINAL = True
RUN_MEDSAM = True      # optional part of the milestone

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/polyp-m2"
    M1_WORK = "/content/drive/MyDrive/polyp-m1"   # Milestone 1 downloads are re-used if present
else:
    WORK, M1_WORK = "/content/work", "/content/none"
DATA = "/content/data"        # unzipped datasets (local disk: fast)
RUNS = f"{WORK}/runs"         # one folder per run: checkpoints, history, test scores (Drive: survives)
MAPS = "/content/m2_maps"     # prediction maps of every run (local)
OUT = "/content/m2"           # tables and figures of this notebook
PVT_REPO = "/content/Polyp-PVT"

for d in [WORK, DATA, RUNS, MAPS, f"{OUT}/results", f"{OUT}/figures"]:
    os.makedirs(d, exist_ok=True)
os.environ.update(WORK=WORK, M1_WORK=M1_WORK, DATA=DATA, PVT_COMMIT=PVT_COMMIT,
                  MEDSAM_COMMIT=MEDSAM_COMMIT, REPO_RAW=REPO_RAW)
print("work folder:", WORK)

In [ ]:
!pip install -q gdown pysodmetrics timm tabulate

Official Polyp-PVT code at the Milestone 1 commit (we use its `PolypPVT` model class unchanged), and our two scripts from this repository.

In [ ]:
%cd /content
!rm -rf Polyp-PVT && git clone -q https://github.com/DengPingFan/Polyp-PVT && git -C Polyp-PVT checkout -q $PVT_COMMIT
!git -C Polyp-PVT log -1 --format='Polyp-PVT %h %cd'
!wget -q -O /content/polyp_m2.py $REPO_RAW/milestone2/polyp_m2.py
!wget -q -O /content/prepare_data.py $REPO_RAW/milestone1/prepare_data.py
!ls -l /content/polyp_m2.py /content/prepare_data.py

## 2. Download data and weights

Same Google Drive links as Milestone 1 (from the official READMEs). `TrainDataset` is needed now.
`PolypPVT.pth` (the authors' trained model) is only used for the pipeline check and as a reference.

In [ ]:
%%bash
set -euo pipefail
mkdir -p $WORK/downloads $WORK/weights
cd $WORK/downloads
for f in TestDataset.zip TrainDataset.zip; do
  [ -f $f ] || [ ! -f $M1_WORK/downloads/$f ] || cp $M1_WORK/downloads/$f .
done
for f in PolypPVT.pth pvt_v2_b2.pth; do
  [ -f $WORK/weights/$f ] || [ ! -f $M1_WORK/weights/$f ] || cp $M1_WORK/weights/$f $WORK/weights/
done
[ -f TestDataset.zip ]  || gdown 1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao -O TestDataset.zip
[ -f TrainDataset.zip ] || gdown 1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb -O TrainDataset.zip
if [ ! -f $WORK/weights/PolypPVT.pth ]; then
  gdown --folder 1xC5Opwu5Afz4xiK5O9v4NnQOZY0A9-2j -O pvt_trained
  find pvt_trained -name '*.pth' -exec cp {} $WORK/weights/PolypPVT.pth \; -quit
fi
if [ ! -f $WORK/weights/pvt_v2_b2.pth ]; then
  gdown --folder 1Eu8v9vMRvt-dyCH0XSV2i77lAd62nPXV -O pvt_backbone
  find pvt_backbone -name 'pvt_v2_b2*.pth' -exec cp {} $WORK/weights/pvt_v2_b2.pth \; -quit
fi
for z in TestDataset.zip TrainDataset.zip; do
  rm -rf $DATA/${z%.zip}
  unzip -q -o $z -d $DATA
done
rm -rf $DATA/__MACOSX
ls -lh $WORK/weights

# the PolypPVT constructor loads the ImageNet backbone from ./pretrained_pth
mkdir -p /content/Polyp-PVT/pretrained_pth
ln -sf $WORK/weights/pvt_v2_b2.pth /content/Polyp-PVT/pretrained_pth/pvt_v2_b2.pth

ni=$(ls $DATA/TrainDataset/image | wc -l); nm=$(ls $DATA/TrainDataset/masks | wc -l)
echo "TrainDataset images=$ni masks=$nm expected=1450"
[ $ni -eq 1450 ] && [ $nm -eq 1450 ] || { echo "train data incomplete" >&2; exit 1; }

## 3. Train / validation / test split

Exactly the Milestone 1 split (`prepare_data.make_split`, seed 0): 10 % of `TrainDataset` held out
for validation, Kvasir images at random, CVC-ClinicDB in blocks of 25 consecutive video frames so a
validation frame is never the neighbour of a training frame. The cell checks it is identical to the
committed `milestone1/results/splits.csv`.

In [ ]:
import sys
sys.path.insert(0, "/content")
import importlib
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import polyp_m2 as m2
import prepare_data
importlib.reload(m2)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
splits = prepare_data.make_split(DATA, val_fraction=0.1, seed=0)
print(splits.groupby(["split", "source"]).size().unstack(fill_value=0), "\n")
n = splits.split.value_counts()
assert (n["train"], n["val"], n["test"]) == (1285, 165, 798), n

m1 = pd.read_csv(f"{REPO_RAW}/milestone1/results/splits.csv")
same = m1.set_index("image").split.sort_index().equals(splits.set_index("image").split.sort_index())
print("identical to the Milestone 1 split:", same)
assert same
splits.to_csv(f"{OUT}/results/splits.csv", index=False)

## 4. Checks before training

**(a) Our inference + metric = the official ones.** `m2.evaluate` re-implements the official
`Test.py` (resize to 352, P1 + P2, upsample, sigmoid, min-max, ×255) and scores with the Milestone 1
metric, in memory. With the authors' `PolypPVT.pth` it must give the Milestone 1 / paper numbers.

In [ ]:
REFERENCE = {"Kvasir": 0.917, "CVC-ClinicDB": 0.937, "CVC-ColonDB": 0.808, "ETIS-LaribPolypDB": 0.787, "CVC-300": 0.900}
authors = m2.build_model(PVT_REPO)
authors.load_state_dict(torch.load(f"{WORK}/weights/PolypPVT.pth", map_location="cpu"))
authors = authors.to(DEVICE)
authors_sum, _ = m2.evaluate(authors, splits[splits.split == "test"], DATA, 352, DEVICE)
authors_sum["paper"] = authors_sum.dataset.map(REFERENCE)
print(authors_sum.round(3).to_string(index=False))
assert (authors_sum.meanDice - authors_sum.paper).abs().max() < 0.0015, "pipeline does not reproduce Milestone 1"
authors_sum.to_csv(f"{OUT}/results/authors_model_check.csv", index=False)
del authors; torch.cuda.empty_cache()

**(b) The official loss is not what the paper describes.** `structure_loss` in `Train.py` calls
`F.binary_cross_entropy_with_logits(pred, mask, reduce='none')`. `reduce` is an old *boolean*
argument; the string `'none'` counts as `True`, so PyTorch returns the **mean** BCE, a single number.
Multiplying that number by the edge weight and dividing by the summed weight gives it back unchanged:
the "weighted BCE" is a plain BCE. Only the weighted-IoU half is edge-weighted. We keep this behaviour
as `structure_official` (the baseline) and test the paper's version as `structure_weighted`.

In [ ]:
import torch.nn.functional as F
logits, mask = torch.randn(2, 1, 64, 64), (torch.rand(2, 1, 64, 64) > 0.7).float()
legacy = F.binary_cross_entropy_with_logits(logits, mask, reduce="none")
print("output shape with reduce='none':", tuple(legacy.shape), "| equals plain mean BCE:",
      torch.allclose(legacy, F.binary_cross_entropy_with_logits(logits, mask)))
for name, fn in m2.LOSSES.items():
    print(f"{name:20s} {fn(logits, mask).item():.4f}")

**(c) Augmentations.** `none` is the official default (`--augmentation False`); `flip_rotate` is the
official option (`RandomRotation(90)` + flips); `strong` adds a zoom-in crop, colour jitter and blur.
Two random draws of each on one training image (mask outline in yellow):

In [ ]:
row = splits[(splits.split == "train") & (splits.source == "Kvasir")].iloc[3]
image = m2.read_rgb(Path(DATA) / row.image)
mask = np.where(cv2.imread(str(Path(DATA) / row["mask"]), 0) > 127, 255, 0).astype(np.uint8)
fig, axes = plt.subplots(2, len(m2.AUGMENTATIONS), figsize=(3.2 * len(m2.AUGMENTATIONS), 6))
for j, (name, fn) in enumerate(m2.AUGMENTATIONS.items()):
    for i in range(2):
        a_img, a_mask = fn(image, mask, np.random.default_rng(10 * j + i + 1))
        axes[i, j].imshow(a_img)
        axes[i, j].contour(a_mask > 127, levels=[0.5], colors="yellow", linewidths=1)
        axes[i, j].set_title(name if i == 0 else "", fontsize=10)
        axes[i, j].axis("off")
plt.tight_layout()
plt.savefig(f"{OUT}/figures/augmentations.png", dpi=120)
plt.show()

## 5. Experiments

One factor at a time against the **baseline = the official recipe**: structure loss as the code
computes it, no augmentation, 352 × 352, multi-scale training (0.75 / 1 / 1.25), AdamW lr 1e-4,
weight decay 1e-4, gradient clamp 0.5, batch 16. Every run: same seed, same split, `EPOCHS` epochs,
the epoch with the best **validation** mean Dice is kept, then scored once on the five test sets.
Mixed precision (fp16) is used to fit the runs into Colab sessions.

Only the baseline also scores the test sets after every epoch (`track_test`), purely to measure in
section 6 how much choosing the epoch on the test set (the official protocol) inflates the result.
It is never used to choose anything.

| Factor | Runs |
|---|---|
| Loss | `structure_official` (baseline), `structure_weighted`, `bce_dice`, `dice` |
| Augmentation | `none` (baseline), `flip`, `flip_rotate`, `strong` |
| Image size (train = test) | 256, **352** (baseline), 448 |

A size-448 run needs more GPU memory; if a T4 runs out, set `"batch_size": 8` for it and say so in the report.

In [ ]:
BASE = dict(loss="structure_official", augmentation="none", size=352, multiscale=True, epochs=EPOCHS)
EXPERIMENTS = {                      # name: change from BASE
    "baseline":                {"track_test": True},
    "loss_structure_weighted": {"loss": "structure_weighted"},
    "loss_bce_dice":           {"loss": "bce_dice"},
    "loss_dice":               {"loss": "dice"},
    "aug_flip":                {"augmentation": "flip"},
    "aug_flip_rotate":         {"augmentation": "flip_rotate"},
    "aug_strong":              {"augmentation": "strong"},
    "size_256":                {"size": 256},
    "size_448":                {"size": 448},
}
FACTORS = {
    "loss": ["baseline", "loss_structure_weighted", "loss_bce_dice", "loss_dice"],
    "augmentation": ["baseline", "aug_flip", "aug_flip_rotate", "aug_strong"],
    "size": ["size_256", "baseline", "size_448"],
}

for name, change in EXPERIMENTS.items():
    if RUN_ONLY and name not in RUN_ONLY:
        continue
    cfg = m2.Config(name=name, **{**BASE, **change})
    m2.train(cfg, splits, DATA, PVT_REPO, RUNS, device=DEVICE, maps_dir=MAPS)
    torch.cuda.empty_cache()

## 6. Results of the experiments

`val` is the validation mean Dice of the chosen epoch (what the choice is based on); the test columns
are mean Dice of that same checkpoint; `mean5` is the plain average over the five test sets.

In [ ]:
TEST = m2.TEST_SETS
res = m2.collect(RUNS, list(EXPERIMENTS))
cols = ["name", "loss", "augmentation", "size", "best_epoch", "best_val_meanDice"] + [f"test_{d}" for d in TEST] + ["test_mean5", "minutes"]
table = res[cols].rename(columns={"best_val_meanDice": "val", **{f"test_{d}": d for d in TEST}, "test_mean5": "mean5"})
table.to_csv(f"{OUT}/results/ablation.csv", index=False)
with open(f"{OUT}/results/ablation.md", "w") as f:
    f.write(table.round(3).to_markdown(index=False))
print(table.round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, len(FACTORS), figsize=(16, 4), sharey=True)
for ax, (factor, names) in zip(axes, FACTORS.items()):
    sub = res.set_index("name").reindex(names).dropna(subset=["best_val_meanDice"])
    x = np.arange(len(sub))
    ax.bar(x - 0.2, sub.best_val_meanDice, 0.4, label="validation (used to choose)")
    ax.bar(x + 0.2, sub.test_mean5, 0.4, label="test, mean of 5 sets")
    ax.set_xticks(x)
    ax.set_xticklabels([str(v) for v in sub[factor]], rotation=15)
    ax.set_title(factor)
    ax.set_ylim(0.6, 1.0)
axes[0].set_ylabel("mean Dice")
axes[0].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/figures/ablation.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name in res.name:
    h = pd.read_csv(f"{RUNS}/{name}/history.csv")
    axes[0].plot(h.epoch, h.train_loss, label=name)
    axes[1].plot(h.epoch, h.val_meanDice, label=name)
axes[0].set_title("training loss (scale 1)"); axes[0].set_xlabel("epoch")
axes[1].set_title("validation mean Dice"); axes[1].set_xlabel("epoch")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"{OUT}/figures/training_curves.png", dpi=150)
plt.show()

**Choosing the epoch on the test set (official protocol) vs on validation (ours).** The baseline
scored the test sets after every epoch. For each test set: the Dice at the epoch chosen on validation,
and the best Dice over all epochs, which is what the official protocol would report.

In [ ]:
h = pd.read_csv(f"{RUNS}/baseline/history.csv")
chosen = h.loc[h.val_meanDice.idxmax()]
h["test_mean5"] = h[[f"test_{d}" for d in TEST]].mean(axis=1)
gap = pd.DataFrame({
    "dataset": TEST + ["mean of 5"],
    "val-chosen epoch (ours)": [chosen[f"test_{d}"] for d in TEST] + [chosen.test_mean5],
    "best epoch on test (official)": [h[f"test_{d}"].max() for d in TEST] + [h.test_mean5.max()],
})
gap["optimism"] = gap.iloc[:, 2] - gap.iloc[:, 1]
gap.to_csv(f"{OUT}/results/protocol_gap.csv", index=False)
print(f"chosen epoch {int(chosen.epoch)}; best mean5 on test at epoch {int(h.loc[h.test_mean5.idxmax(), 'epoch'])}")
print(gap.round(4).to_string(index=False))

plt.figure(figsize=(9, 4))
for d in TEST:
    plt.plot(h.epoch, h[f"test_{d}"], lw=1, label=d)
plt.plot(h.epoch, h.val_meanDice, "k", lw=2, label="validation")
plt.axvline(chosen.epoch, color="k", ls="--", lw=1)
plt.xlabel("epoch"); plt.ylabel("mean Dice"); plt.title("baseline: validation and test Dice per epoch (dashed: chosen epoch)")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(f"{OUT}/figures/protocol_baseline.png", dpi=150)
plt.show()

## 7. Final model

For each factor take the option with the highest **validation** Dice, but only if it beats the
baseline option by at least `MIN_GAIN` (smaller differences are within run-to-run noise with one
seed). Train it for `FINAL_EPOCHS`, together with the official recipe for the same length
(`baseline_long`), so the comparison is fair. Both keep the validation-best epoch.

In [ ]:
MIN_GAIN = 0.005
v = res.set_index("name").best_val_meanDice
final_change = {}
for factor, names in FACTORS.items():
    done = [n for n in names if n in v.index]
    if not done or "baseline" not in v.index:
        continue
    best = max(done, key=lambda n: v[n])
    if best != "baseline" and v[best] - v["baseline"] >= MIN_GAIN:
        final_change[factor] = res.set_index("name").loc[best, factor]
    print(f"{factor:12s} best on validation: {best:24s} ({v[best]:.4f} vs baseline {v['baseline']:.4f})")
print("final settings, change from the official recipe:", final_change or "none")

FINAL_RUNS = {"baseline_long": {}, "final": final_change}
if RUN_FINAL:
    for name, change in FINAL_RUNS.items():
        cfg = m2.Config(name=name, **{**BASE, "epochs": FINAL_EPOCHS, **change})
        if name == "final" and not change:
            print("no factor improved on validation: 'final' is the same as 'baseline_long', skipped")
            continue
        m2.train(cfg, splits, DATA, PVT_REPO, RUNS, device=DEVICE, maps_dir=MAPS)

Final comparison on the five test sets: our two retrained models (validation-chosen), the authors'
released model (test-chosen, section 4) and PraNet (Milestone 1).

In [ ]:
PRANET_M1 = {"Kvasir": 0.894, "CVC-ClinicDB": 0.899, "CVC-ColonDB": 0.712, "ETIS-LaribPolypDB": 0.628, "CVC-300": 0.871}
rows = [{"model": "PraNet, authors (Milestone 1)", **PRANET_M1},
        {"model": "Polyp-PVT, authors (epoch chosen on test)", **dict(zip(authors_sum.dataset, authors_sum.meanDice))}]
LABELS = {"baseline_long": "Polyp-PVT, official recipe, ours (val)", "final": "Polyp-PVT, improved, ours (val)"}
for _, r in m2.collect(RUNS, list(LABELS)).iterrows():
    rows.append({"model": LABELS[r["name"]], **{d: r[f"test_{d}"] for d in TEST}})
final_table = pd.DataFrame(rows)
final_table["mean5"] = final_table[TEST].mean(axis=1)
final_table.to_csv(f"{OUT}/results/final_comparison.csv", index=False)
with open(f"{OUT}/results/final_comparison.md", "w") as f:
    f.write(final_table.round(3).to_markdown(index=False))
print(final_table.round(3).to_string(index=False))

ax = final_table.set_index("model")[TEST].T.plot.bar(figsize=(12, 4), width=0.8)
ax.set_ylim(0.5, 1.0); ax.set_ylabel("mean Dice"); ax.tick_params(axis="x", rotation=0)
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.savefig(f"{OUT}/figures/final_comparison.png", dpi=150)
plt.show()

## 8. Optional: MedSAM with bounding-box prompts

[MedSAM](https://github.com/bowang-lab/MedSAM) is SAM fine-tuned on 1.5 M medical image–mask pairs. It
does not find the polyp itself: it segments what is inside a box it is given. Three settings:

- **GT box**: one box per polyp region, from the ground-truth mask, as an endoscopist drawing a tight
  box would give. This is *semi-automatic* and not comparable one-to-one with Polyp-PVT: the box
  already tells MedSAM where and how big the polyp is.
- **GT box ± 10 %**: every side moved by up to 10 % of the box size, a less careful box.
- **Polyp-PVT box**: boxes from our final Polyp-PVT prediction (threshold 0.5), a fully automatic
  two-stage pipeline: does MedSAM refine Polyp-PVT's masks?

MedSAM outputs almost binary maps, so its threshold-averaged Dice is close to its Dice@0.5.

In [ ]:
%%bash
set -e
cd /content
[ -d MedSAM ] || (git clone -q https://github.com/bowang-lab/MedSAM && git -C MedSAM checkout -q $MEDSAM_COMMIT)
mkdir -p $WORK/weights
if [ ! -f $WORK/weights/medsam_vit_b.pth ]; then
  gdown --folder 1ETWmi4AiniJeWOt6HAsYgTjYv_fkgzoN -O /content/medsam_ckpt
  find /content/medsam_ckpt -name 'medsam_vit_b.pth' -exec cp {} $WORK/weights/ \; -quit
fi
ls -lh $WORK/weights/medsam_vit_b.pth

In [ ]:
if RUN_MEDSAM:
    medsam = m2.load_medsam("/content/MedSAM", f"{WORK}/weights/medsam_vit_b.pth", DEVICE)
    test_rows = splits[splits.split == "test"]
    box_run = "final" if Path(f"{MAPS}/final").exists() else "baseline_long" if Path(f"{MAPS}/baseline_long").exists() else "baseline"
    settings = {   # short name: (label, arguments)
        "medsam_gt_box": ("MedSAM, GT box", dict(box_source="gt")),
        "medsam_gt_box_jitter10": ("MedSAM, GT box +-10%", dict(box_source="gt", jitter=0.1)),
        "medsam_pvt_box": (f"MedSAM, Polyp-PVT box ({box_run})", dict(box_source="pred", pred_maps=f"{MAPS}/{box_run}")),
    }
    med_rows = []
    for short, (label, kw) in settings.items():
        s, per_image = m2.evaluate_medsam(medsam, test_rows, DATA, DEVICE, save_dir=f"{MAPS}/{short}", **kw)
        per_image.to_csv(f"{OUT}/results/{short}_per_image.csv", index=False)
        med_rows.append({"model": label, **dict(zip(s.dataset, s.meanDice))})
        print(label, s[["dataset", "meanDice", "meanIoU", "Dice@0.5"]].round(3).to_dict("records"))
    medsam_table = pd.concat([final_table, pd.DataFrame(med_rows)], ignore_index=True)
    medsam_table["mean5"] = medsam_table[TEST].mean(axis=1)
    medsam_table.to_csv(f"{OUT}/results/medsam_comparison.csv", index=False)
    with open(f"{OUT}/results/medsam_comparison.md", "w") as f:
        f.write(medsam_table.round(3).to_markdown(index=False))
    print(medsam_table.round(3).to_string(index=False))

Keep tables, figures and the final prediction maps in the work folder (Drive).

In [ ]:
!rm -rf $WORK/m2_out && mkdir -p $WORK/m2_out && cp -r /content/m2/results /content/m2/figures $WORK/m2_out/
!cd /content/m2_maps && zip -qr $WORK/m2_out/maps.zip . && ls -lh $WORK/m2_out $WORK/m2_out/results

## Conclusion

Fill in from the tables above (the write-up template is `milestone2/README.md`):

- validation-chosen retraining of the official recipe vs the authors' test-chosen model, and how large
  the optimism of the official protocol is (section 6);
- which loss, augmentation and image size helped on validation, and whether the test sets agree;
- the final model against the official recipe, the authors' model and PraNet;
- MedSAM: how much a (perfect / imperfect / predicted) box helps.